## Importing Libraries

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import clickhouse_connect

### Define Parameters

In [2]:
MONTH = pd.Period("2026-07")  # <-- only thing to change each run

START, END = MONTH.start_time.date(), (MONTH + 1).start_time.date()

def date_filter(col: str) -> str:
    """SQL WHERE clause for the analysis month, on the given timestamp column."""
    return f"WHERE {col} >= '{START}' AND {col} < '{END}'"

# KYC dump is for the same month as the IMEI data
KYC_TAG = MONTH.strftime("%m_%y")          # e.g. "07_26"
MONTH_TAG = MONTH.strftime("%Y-%m")        # e.g. "2026-07"

KYC_DIR = Path("/Volumes/E$/KYC/Merged Clean Dumps/2026")
OUT_DIR = Path("/Volumes/E$/CEIR/Clean Dumps")
MCC_CSV = Path("/Users/wmuheki/Documents/Projects/Analytics/ceir/clean_dumps/MCC_Each_country.csv")

print(date_filter("imei_first_seen"))
print(f"KYC files: df_{KYC_TAG}.parquet / df_NID_{KYC_TAG}.parquet")

WHERE imei_first_seen >= '2026-07-01' AND imei_first_seen < '2026-08-01'
KYC files: df_07_26.parquet / df_NID_07_26.parquet


### Define Clickhouse Connect - Single Reused

In [3]:
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    settings={
        'max_memory_usage': 4000000000,  # 4GB max
        'max_threads': 2,
        'priority': 5
    }
)

### Helper Functions

In [4]:
def attach_kyc(target_df: pd.DataFrame, nid_df: pd.DataFrame, fallback_df: pd.DataFrame) -> pd.DataFrame:
    """Attach KYC on msisdn: NID-registered values win, fallback fills the gaps."""
    out = target_df.merge(nid_df, on='msisdn', how='left')
    out = out.merge(fallback_df, on='msisdn', how='left', suffixes=('', '_fb'))

    overlap = ['id_type', 'id_number', 'prefix', 'mno']  # columns present in both KYC frames
    for col in overlap:
        out[col] = out[col].astype('object').fillna(out[f'{col}_fb'].astype('object'))

    out = out.drop(columns=[f'{col}_fb' for col in overlap])

    for col in ['id_type', 'prefix', 'mno']:
        out[col] = out[col].astype('category')
    return out


IMSI_PREFIX_MNO = {
    '64110': 'MTN',
    '64120': 'HAMILTON',
    '64108': 'TALKIO',
    '64101': 'AIRTEL',
    '64122': 'AIRTEL',
}

def fill_mno_from_imsi(df: pd.DataFrame) -> pd.DataFrame:
    """Where mno is missing, derive it from the IMSI prefix."""
    mask = df['mno'].isna()
    imsi = df['imsi'].astype('string')

    if isinstance(df['mno'].dtype, pd.CategoricalDtype):
        new_cats = [m for m in set(IMSI_PREFIX_MNO.values()) if m not in df['mno'].cat.categories]
        if new_cats:
            df['mno'] = df['mno'].cat.add_categories(new_cats)

    for prefix, operator in IMSI_PREFIX_MNO.items():
        df.loc[mask & imsi.str.startswith(prefix), 'mno'] = operator
    return df


def add_country(df: pd.DataFrame, mcc_lookup: pd.DataFrame) -> pd.DataFrame:
    """Extract MCC (first 3 digits of IMSI) and merge in the country name."""
    df['imsi'] = df['imsi'].astype('string')
    df['mcc'] = (
        df['imsi']
        .str.replace(r'\D+', '', regex=True)  # keep digits only
        .str.slice(0, 3)
    )
    df = df.merge(mcc_lookup[['mcc', 'country']], how='left', on='mcc')
    df['country'] = df['country'].fillna('UNKNOWN')
    return df


def save_monthly_parquet(df: pd.DataFrame, category: str) -> Path:
    """Save df to <OUT_DIR>/<category>/<category.lower()>_<YYYY-MM>.parquet."""
    out_path = OUT_DIR / category / f"{category.lower()}_{MONTH_TAG}.parquet"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_path)
    print(f"Saved {len(df):,} rows -> {out_path}")
    return out_path

### Importing KYC data

In [5]:
df = pd.read_parquet(KYC_DIR / f"df_{KYC_TAG}.parquet")
df_NID = pd.read_parquet(KYC_DIR / f"df_NID_{KYC_TAG}.parquet")

In [6]:
df.head()

,msisdn,first_name,surname,id_type,id_number,prefix,mno
0,0770811460,STELLA,ARINAITWE,NATIONAL_ID,CF82027108KCZK,077,MTN
1,0770811504,RONALD,BULIME,NATIONAL_ID,CM89023100F8YK,077,MTN
2,0770811551,MERETH ROSE,NABUGUBE,NATIONAL_ID,CF970511012F3C,077,MTN
3,0770811565,AMIDU,ALIONZI,NATIONAL_ID,CM8806610254VG,077,MTN
4,0770811592,THEOPHILE MUHAWE,NDAGENGWA,REFUGEE_ID,FVA-00057937,077,MTN


In [7]:
df_NID.head()

,msisdn,first_name,surname,id_type,id_number,prefix,mno,gender,birth_year,age,district
0,0770811460,STELLA,ARINAITWE,NATIONAL_ID,CF82027108KCZK,077,MTN,Female,1982,44,MBARARA
1,0770811504,RONALD,BULIME,NATIONAL_ID,CM89023100F8YK,077,MTN,Male,1989,37,LUWEERO
2,0770811551,MERETH ROSE,NABUGUBE,NATIONAL_ID,CF970511012F3C,077,MTN,Female,1997,29,SIRONKO
3,0770811565,AMIDU,ALIONZI,NATIONAL_ID,CM8806610254VG,077,MTN,Male,1988,38,KOBOKO
4,0770811594,JOHN,BALUKU,NATIONAL_ID,CM86015102NMTJ,077,MTN,Male,1986,40,KASESE


In [8]:
# Keep only local-format MSISDNs (10 digits, leading 0), then convert to 256 format.
# NOTE: this deliberately drops rows already in international 256... format (12 digits);
# widen the filter to .isin([10, 12]) if those should be kept.
df = df[df['msisdn'].astype(str).str.len() == 10].copy()
df_NID = df_NID[df_NID['msisdn'].astype(str).str.len() == 10].copy()

df['msisdn'] = df['msisdn'].astype(str).str.replace(r'^0', '256', regex=True)
df_NID['msisdn'] = df_NID['msisdn'].astype(str).str.replace(r'^0', '256', regex=True)

# Drop unnecessary columns to save memory
df = df.drop(columns=['surname', 'first_name'])
df_NID = df_NID.drop(columns=['surname', 'first_name'])

### Importing MCC Lookup (loaded once, shared by all datasets)

In [9]:
mcc_lu = pd.read_csv(MCC_CSV, dtype=str)

mcc_lu = mcc_lu.rename(columns={"MCC": "mcc", "Country": "country"})
mcc_lu["mcc"] = mcc_lu["mcc"].astype("string").str.strip()
mcc_lu["country"] = mcc_lu["country"].astype("string").str.strip()
mcc_lu = mcc_lu.drop_duplicates(subset=["mcc"])

mcc_lu.head()

,mcc,country
0,289,Abkhazia
1,412,Afghanistan
2,276,Albania
3,603,Algeria
4,544,American Samoa


### Importing GSMA data

In [10]:
gsma_query = """
SELECT
    tac,
    oem,
    brand,
    model,
    marketing_name,
    device_type,
    os_family,
    os_version,
    sim_slots,
    has_2g,
    has_3g,
    has_4g,
    has_5g,
    year_released
FROM ceir.gsma_devices
"""
gsma_df = client.query_df(gsma_query)
len(gsma_df)

290402

In [11]:
gsma_df.head()

,tac,oem,brand,model,marketing_name,device_type,os_family,os_version,sim_slots,has_2g,has_3g,has_4g,has_5g,year_released
0,35697403,Not Known,Not Known,ROWEL K658,,Handheld,,,0,0,0,0,0,0
1,35697404,Not Known,G crown,"G265, G765, G865, G965",,Handheld,,,0,0,0,0,0,0
2,35697405,Not Known,QMobile,Q4,Q4 TV,Handheld,Other,,0,1,0,0,0,2013
3,35697406,Not Known,Apple,iPad mini (A1600),iPad mini 3,Tablet,iOS,8_1,1,1,1,1,0,2014
4,35697407,Not Known,TC,TC F6,,Mobile Phone/Feature phone,,,0,0,0,0,0,0


### Importing Fake IMEIs Table

Fake IMEIs are intentionally **not** merged with GSMA: their TACs are typically
unallocated, so the merge would produce mostly nulls/noise.

In [12]:
fake_query = f"SELECT * FROM ceir_gold.imeis_fake_v {date_filter('imei_first_seen')}"
fake_df = client.query_df(fake_query)
len(fake_df)

543620

In [13]:
fake_df.head()

,imei,last_seen,imei_status,imsi,msisdn,device_type,imei_first_seen,cgi,rat,core_type
0,00000000000045,2026-07-16 18:50:47,W,641010264301569,,<NA>,2026-07-06 08:27:38,641-01-1210-25783,1,1
1,00000000002346,2026-08-02 08:43:36,W,641010260961163,,<NA>,2026-07-31 06:31:26,641-01-4014-19922,1,1
2,00000003421213,2026-07-20 00:51:42,W,641010431759756,256731784083,<NA>,2026-07-09 18:09:25,641-01-3500-55976,1,1
3,00000357769974,2026-08-02 18:45:14,W,641010421158768,,<NA>,2026-07-02 02:07:32,641-10-1234-5678,6,2
4,00000741887190,2026-07-25 10:24:38,W,641010422602885,,<NA>,2026-07-02 10:14:54,641-01-6013-50873,1,1


In [14]:
# Drop empty device_type column and re-arrange columns
fake_df = fake_df.drop(columns=['device_type'])
fake_df = fake_df[['imei_first_seen', 'last_seen', 'imei', 'imei_status', 'imsi', 'msisdn', 'rat', 'cgi']]

In [15]:
# Enrich: KYC coalesce -> MNO from IMSI where missing -> country from MCC
fake_df = attach_kyc(fake_df, df_NID, df)
fake_df = fill_mno_from_imsi(fake_df)
fake_df = add_country(fake_df, mcc_lu)

In [16]:
fake_df.head()

,imei_first_seen,last_seen,imei,imei_status,imsi,msisdn,rat,cgi,id_type,id_number,prefix,mno,gender,birth_year,age,district,mcc,country
0,2026-07-06 08:27:38,2026-07-16 18:50:47,00000000000045,W,641010264301569,,1,641-01-1210-25783,NaN,<NA>,NaN,AIRTEL,NaN,<NA>,<NA>,NaN,641,Uganda
1,2026-07-31 06:31:26,2026-08-02 08:43:36,00000000002346,W,641010260961163,,1,641-01-4014-19922,NaN,<NA>,NaN,AIRTEL,NaN,<NA>,<NA>,NaN,641,Uganda
2,2026-07-09 18:09:25,2026-07-20 00:51:42,00000003421213,W,641010431759756,256731784083,1,641-01-3500-55976,NATIONAL_ID,CF90072103VYEK,0731,AIRTEL,Female,1990,36,BUDAKA,641,Uganda
3,2026-07-02 02:07:32,2026-08-02 18:45:14,00000357769974,W,641010421158768,,6,641-10-1234-5678,NaN,<NA>,NaN,AIRTEL,NaN,<NA>,<NA>,NaN,641,Uganda
4,2026-07-02 10:14:54,2026-07-25 10:24:38,00000741887190,W,641010422602885,,1,641-01-6013-50873,NaN,<NA>,NaN,AIRTEL,NaN,<NA>,<NA>,NaN,641,Uganda


### Importing Genuine IMEIs Table

In [17]:
genuine_query = f"SELECT * FROM ceir_gold.imeis_genuine_v {date_filter('imei_first_seen')}"
genuine_df = client.query_df(genuine_query)
len(genuine_df)

3340011

In [18]:
# Drop device_type — regenerated from GSMA TAC data below
genuine_df = genuine_df.drop(columns=['device_type'])

# Derive TAC (first 8 digits of the IMEI) and re-arrange columns
genuine_df['tac'] = genuine_df['imei'].astype(str).str[:8]
genuine_df = genuine_df[['imei_first_seen', 'last_seen', 'tac', 'imei', 'imei_status', 'imsi', 'msisdn', 'rat', 'cgi']]

# Merge with GSMA data to get device details
genuine_df = genuine_df.merge(gsma_df, on='tac', how='left')

In [19]:
# Enrich: KYC coalesce -> MNO from IMSI where missing -> country from MCC
genuine_df = attach_kyc(genuine_df, df_NID, df)
genuine_df = fill_mno_from_imsi(genuine_df)
genuine_df = add_country(genuine_df, mcc_lu)

In [20]:
# Convert GSMA numeric columns to nullable integers (removes decimal points)
int_cols = ['sim_slots', 'has_2g', 'has_3g', 'has_4g', 'has_5g', 'year_released']
genuine_df[int_cols] = genuine_df[int_cols].astype('Int64')

In [21]:
genuine_df.head()

,imei_first_seen,last_seen,tac,imei,imei_status,imsi,msisdn,rat,cgi,oem,...,id_type,id_number,prefix,mno,gender,birth_year,age,district,mcc,country
0,2026-07-07 05:44:17,2026-08-11 08:11:17,35530149,35530149594175,W,630860226067872,,6,641-10-1234-5678,INFINIX TECHNOLOGY LIMITED,...,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,630,Democratic Republic of Congo
1,2026-07-02 20:46:42,2026-08-02 18:21:12,35530149,35530149769995,W,641101016884088,256786266427,4,641-10-1234-5678,INFINIX TECHNOLOGY LIMITED,...,NATIONAL_ID,CF840341079M7H,078,MTN,Female,1984,42,NTUNGAMO,641,Uganda
2,2026-07-02 14:49:15,2026-08-11 20:15:49,35530149,35530149794665,W,639035077369780,254734224382,6,641-01-4012-17183,INFINIX TECHNOLOGY LIMITED,...,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,639,Kenya
3,2026-07-06 23:25:49,2026-08-03 08:07:01,35530149,35530149834880,W,641010414384681,256741548775,1,641-01-1220-43751,INFINIX TECHNOLOGY LIMITED,...,NATIONAL_ID,CF000581094Q8A,074,AIRTEL,Female,2000,26,AMURIA,641,Uganda
4,2026-07-01 17:05:20,2026-08-02 16:54:31,35530149,35530149838763,W,641101974701285,,4,641-10-1234-5678,INFINIX TECHNOLOGY LIMITED,...,NaN,<NA>,NaN,MTN,NaN,<NA>,<NA>,NaN,641,Uganda


### Cloned IMEIs

In [22]:
# Filter in SQL so only the analysis month is pulled (the view's timestamp
# column is first_detected_at, renamed to imei_first_seen after loading)
clone_query = f"SELECT * FROM ceir_gold.cloned_imeis_v {date_filter('first_detected_at')}"
clone_df = client.query_df(clone_query)
len(clone_df)

193865

In [23]:
# Drop device_type (regenerated from GSMA) and the array columns
clone_df = clone_df.drop(columns=['device_type', 'imsis', 'msisdns'])

# Rename for consistency with the other datasets
clone_df = clone_df.rename(columns={'first_detected_at': 'imei_first_seen', 'last_change_at': 'last_seen'})

# Derive TAC and re-arrange columns
clone_df['tac'] = clone_df['imei'].astype(str).str[:8]
clone_df = clone_df[['imei_first_seen', 'last_seen', 'tac', 'imei', 'imsi_count', 'msisdn_count']]

# Merge with GSMA data to get device details
clone_df = clone_df.merge(gsma_df, on='tac', how='left')

In [24]:
clone_df.head()

,imei_first_seen,last_seen,tac,imei,imsi_count,msisdn_count,oem,brand,model,marketing_name,device_type,os_family,os_version,sim_slots,has_2g,has_3g,has_4g,has_5g,year_released
0,2026-07-15 22:55:34,2026-08-02 12:43:10,86333307,86333307060390,2,2,Mobiwire Mobiles (Ningbo) Co Ltd,Sagetel,H5111L,Alola 4G,Smartphone,Android,14,2.0,1.0,1.0,1.0,0.0,2024.0
1,2026-07-17 09:56:30,2026-08-02 07:17:44,35015114,35015114989166,3,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-07-13 14:16:36,2026-08-02 07:52:28,35577543,35577543376176,4,4,Samsung Korea,Samsung,SM-A055F/DS,Galaxy A05,Smartphone,Android,13,2.0,1.0,1.0,1.0,0.0,2023.0
3,2026-07-18 23:03:09,2026-08-03 02:50:58,35585533,35585533816679,3,3,Tecno Telecom (HK) Limited,TECNO,T467,,Mobile Phone/Feature phone,,,3.0,1.0,0.0,0.0,0.0,2021.0
4,2026-07-19 10:11:20,2026-07-30 16:02:04,35184491,35184491900502,3,4,Samsung Korea,Samsung,SM-A055F/DS,Galaxy A05,Smartphone,Android,13,2.0,1.0,1.0,1.0,0.0,2023.0


### Export — filenames derived from MONTH, nothing to edit

In [25]:
save_monthly_parquet(fake_df, "Fake")
save_monthly_parquet(genuine_df, "Genuine")
save_monthly_parquet(clone_df, "Cloned")

Saved 543,620 rows -> /Volumes/E$/CEIR/Clean Dumps/Fake/fake_2026-07.parquet
Saved 3,446,671 rows -> /Volumes/E$/CEIR/Clean Dumps/Genuine/genuine_2026-07.parquet
Saved 197,375 rows -> /Volumes/E$/CEIR/Clean Dumps/Cloned/cloned_2026-07.parquet


PosixPath('/Volumes/E$/CEIR/Clean Dumps/Cloned/cloned_2026-07.parquet')

### Cleanup

In [26]:
client.close()